# ML-Based AS Tagging — Example Notebook

This notebook demonstrates how to use the **supervised ML tagging** module integrated into the AS Tagging Toolkit.

## Overview

The ML module supports **4 models**:
| Model | Type | Requires Topology |
|---|---|---|
| XGBoost | Tree ensemble | ❌ |
| MLP | Neural network | ❌ |
| GraphConv | Graph Neural Network | ✅ (CAIDA AS-rel) |
| APPNP | MLP + PageRank | ✅ (CAIDA AS-rel) |

The pipeline automatically:
1. Engineers features from the snapshot atomic tags
2. Builds AS topology graph (if data available)
3. Runs stratified K-fold cross-validation on all models
4. Selects the best model by your chosen metric
5. Retrains on all labeled data and assigns predictions

---

## Installation

```bash
# Base package
pip install as-tagging

# ML extras (required for this notebook) — pins torch 2.3.0 + torchdata 0.7.1 for DGL
pip install as-tagging[ml]
# or manually (graph models need torch 2.3.0 + torchdata 0.7.1 + pydantic on macOS):
pip install torch==2.3.0 torchdata==0.7.1 pydantic "typing-extensions>=4.14.0" xgboost scikit-learn numpy
pip install dgl -f https://data.dgl.ai/wheels/torch-2.3/repo.html
```

In [ ]:
# --- DGL troubleshooting: run this if you see graphbolt or DILL_AVAILABLE errors ---
import sys
print("Python:", sys.executable)
import torch
print("PyTorch:", torch.__version__)
# Patch for PyTorch 2.3.0 + torchdata: DILL_AVAILABLE was removed from torch
_common = torch.utils.data.datapipes.utils.common
if not hasattr(_common, "DILL_AVAILABLE"):
    try:
        _common.DILL_AVAILABLE = torch.utils._import_utils.dill_available()
    except (AttributeError, ImportError):
        try:
            import dill
            _common.DILL_AVAILABLE = True
        except ImportError:
            _common.DILL_AVAILABLE = False
try:
    import dgl
    print("DGL:", dgl.__version__)
    import dgl.graphbolt  # This triggers the graphbolt load
    print("GraphBolt: OK")
except Exception as e:
    print("Error:", e)
    print("\nFix: In a terminal, activate the SAME env as this notebook, then:")
    print("  pip install pydantic \"typing-extensions>=4.14.0\"")
    print("  # If graphbolt errors: pip uninstall torch torchdata dgl -y")
    print("  #   pip install torch==2.3.0 torchdata==0.7.1")
    print("  #   pip install dgl -f https://data.dgl.ai/wheels/torch-2.3/repo.html")
    print("Restart the kernel and re-run.")

---
## Step 1: Load Snapshot Data

You need an AS feature snapshot. Choose **either** Online or Offline mode.

### Option A: Online Mode (HuggingFace)

In [ ]:
from as_tagging import ASTagging, OnlineSnapshotProvider
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Use HF_TOKEN env var or provide directly
online_provider = OnlineSnapshotProvider(token=os.environ.get("HF_TOKEN"))

# Pick a date
DATE = "2025-01"
# use_cache=False forces a fresh Hub download + extract (avoids stale cached parquet under ~/.cache/as_tagging/).
tagger = ASTagging(snapshot_provider=online_provider, date=DATE, use_cache=False)
print(f"Loaded {len(tagger.atomic_tags)} ASNs from HuggingFace")

### Option B: Offline Mode (Local Files)

In [ ]:
from as_tagging import ASTagging, OfflineSnapshotProvider

DATA_PATH = fr"../as_feature_zenodo" # Update to your path
DATE = "2026-01"

offline_provider = OfflineSnapshotProvider(DATA_PATH)
# use_cache=False re-extracts from the tarball so cached parquet matches the Zenodo folder (avoids stale ~/.cache/as_tagging/<DATE>/).
tagger = ASTagging(snapshot_provider=offline_provider, date=DATE, use_cache=False)
print(f"Loaded {len(tagger.atomic_tags)} ASNs from local files")

In [ ]:
from as_tagging import ASTagging, OfflineSnapshotProvider

offline_provider = OfflineSnapshotProvider(
    "../as_feature_zenodo",
    package_overrides={
        "2026-01": "ii.as-feature-snapshot.2026-01_ipinfo_geo.tar.gz",
    },
)

# Optional: confirm which tarball will be used
print(offline_provider.get_package_name("2026-01"))

tagger = ASTagging(
    snapshot_provider=offline_provider,
    date="2026-01",
    use_cache=False,  # recommended when switching variants the first time
)

---
## Step 2: Prepare Labeled Data

ML tagging requires **labeled ASNs** — a list of positive examples (ASNs with the property) and negative examples (ASNs without it).

### Prepare your training data

Suppose we want to tag **residential ISPs** versus **non-residential eyeball ASes**.

In [ ]:
# # Uncomment the following lines to use a small set of positive and negative examples if you can't download the positive and negative Access ASNs used in the paper
# positive_asns = [
#     # US
#     "7018",   # AT&T
#     "7922",   # Comcast
#     "20001",  # Charter (Spectrum)
#     "701",    # Verizon
#     "22773",  # Cox
#     "20115",  # Charter
#     "7843",   # Charter
#     "10796",  # Time Warner Cable
#     "5650",   # Frontier
#     # EU
#     "3320",   # Deutsche Telekom
#     "5410",   # BT
#     "12322",  # Free (France)
#     "6830",   # Liberty Global
#     "6805",   # Telefonica Germany
#     # APAC
#     "4134",   # China Telecom
#     "4837",   # China Unicom
#     "17676",  # SoftBank (Japan)
#     "9808",   # China Mobile
#     "4766",   # Korea Telecom
# ]

# # Negative examples: content/cloud/CDN providers (clearly NOT residential)
# negative_asns = [
#     "13335",  # Cloudflare
#     "15169",  # Google
#     "32934",  # Facebook (Meta)
#     "16509",  # Amazon AWS
#     "8075",   # Microsoft
#     "20940",  # Akamai
#     "54113",  # Fastly
#     "14618",  # Amazon
#     "396982", # Google Cloud
#     "36492",  # Google
#     "2906",   # Netflix
#     "714",    # Apple
#     "46489",  # Twitch
#     "19551",  # Incapsula
#     "13414",  # Twitter
#     "63179",  # Valve
#     "14907",  # Wikipedia
#     "36351",  # SoftLayer (IBM Cloud)
#     "19281",  # Quad9 DNS
# ]

# Please download the positive and negative ASNs from the following link and put them in the `data` folder:
# TODO: add link here
import json

with open("./data/residential_isp/all_residential_eyeball_asns.json") as f:
    positive_asns = json.load(f)["asns"]

with open("./data/residential_isp/all_non_residential_eyeball_asns.json") as f:
    negative_asns = json.load(f)["asns"]

# with open("./data/old_data/access_positive_asns.json") as f:
#     positive_asns = json.load(f)

# with open("./data/old_data/access_negative_asns.json") as f:
#     negative_asns = json.load(f)

print(f"Prepared {len(positive_asns)} positive (residential) and {len(negative_asns)} negative (non-residential but eyeball) ASNs")

Since we only differentiate residential from eyeball ASes, we need to collect a list of eyeball ASes as the prediction scope.

In [ ]:
# Obtain AS list with or without a certain tag; use either of the following two variables

# Option 1: obtain AS list without a certain tag
eyeball_asns = tagger.ListASNsWithoutTag("No Eyeball", treat_false_as_missing=True)

# Option 2: manually assign a tag to eyeball ASes
tagger.AssignTag(
    tag_name="Eyeball",
    expression=lambda tags: (tags.get('apnic-eyeball_eyeball_cnt', 0) > 0))

eyeball_asns_test = tagger.FetchTag("Eyeball")
print(set(eyeball_asns_test.keys())==set(eyeball_asns))

# Keep only ASNs that have eyeballs in pos/neg for ML training
from as_tagging import normalize_asn_input
eyeball_canon = set(normalize_asn_input(a) for a in eyeball_asns)
positive_asns = [a for a in positive_asns if normalize_asn_input(a) in eyeball_canon]
negative_asns = [a for a in negative_asns if normalize_asn_input(a) in eyeball_canon]
print(f"Filtered to eyeball ASes only: {len(positive_asns)} positive, {len(negative_asns)} negative")

### Inspect the ML feature table (`build_feature_dataframe`)

Same pipeline as **`MLTagger` / `AssignMLTag`**: infer numerical vs categorical columns (from `schema.json` / manifest when available, else heuristics), then build the matrix with **log1p counts**, **default share columns**, and **`drop_numerators=True`**. The **`asn`** column is not used as a feature (identifier). Inspect `feature_df`, `num_cols`, and `cat_cols` below before running training.


In [ ]:
from as_tagging.ml.feature_engineering import (
    identify_feature_types,
    identify_feature_types_from_metadata,
    build_feature_dataframe,
)

manifest = None
if hasattr(tagger.snapshot_provider, "get_manifest"):
    try:
        manifest = tagger.snapshot_provider.get_manifest(DATE)
    except Exception:
        pass

snapshot_schema = None
if hasattr(tagger.snapshot_provider, "get_schema"):
    try:
        snapshot_schema = tagger.snapshot_provider.get_schema(DATE)
    except Exception:
        pass

num_feats, cat_feats, typing_src = identify_feature_types_from_metadata(
    manifest=manifest, schema=snapshot_schema
)
if not num_feats and not cat_feats:
    num_feats, cat_feats = identify_feature_types(tagger.atomic_tags)
    typing_src = "heuristic_snapshot_values"
feature_df, num_cols, cat_cols = build_feature_dataframe(
    tagger.atomic_tags,
    numerical_features=num_feats,
    categorical_features=cat_feats,
    share_specs=None,
    drop_numerators=True,
)

print(f"Feature typing source: {typing_src}")
print(f"Raw typed columns: {len(num_feats)} numerical, {len(cat_feats)} categorical")
print(f"build_feature_dataframe → shape {feature_df.shape}")
print(f"num_cols in matrix ({len(num_cols)}), all:")
print("\n".join(num_cols))
print(f"cat_cols in matrix ({len(cat_cols)}): {cat_cols}")
feature_df.iloc[:3, :min(8, feature_df.shape[1])]


---
## Step 3: Train, Cross-Validate & Auto-Select (One-Line API)

The simplest way to do ML tagging — `AssignMLTag()` handles everything:

1. Feature engineering
2. Graph construction (if topology available)
3. K-fold CV on all requested models
4. Best model selection
5. Final training and tag assignment

In [ ]:
results = tagger.AssignMLTag(
    tag_name="Residential ISP",            # Name of the new composite tag
    positive_asns=positive_asns,            # ASNs WITH the property
    negative_asns=negative_asns,            # ASNs WITHOUT the property
    models=None,                            # None = try all 4 models
    n_folds=5,                              # 5-fold stratified CV
    metric="f1",                            # Select best model by F1
    threshold=0.5,                          # Probability cutoff for True/False
    verbose=True,                           # Print progress
    asns=eyeball_asns,                      # assign tag to only eyeball ASNs
    keep_training_labels=True,              # True = keep literal labels for train ASNs to ensure most accurate tags
    xgboost_profile="stochastic",           # Use old pipeline profile; reduces fold variance
)

---
## Step 4: Inspect Results

### CV Comparison Table

In [ ]:
# Show the cross-validation comparison table
cv_df = results["cv_results"]
print(f"Best model: {results['best_model']}\n")

# Display key columns
display_cols = ["model", "f1_mean", "f1_std", "precision_mean", "recall_mean", "auroc_mean"]
display_cols = [c for c in display_cols if c in cv_df.columns]
cv_df[display_cols].round(3)

### Per-Fold Details

In [ ]:
# Check per-fold results for the best model
import pandas as pd

best = results["best_model"]
fold_df = pd.DataFrame(results["cv_raw"][best])
print(f"Per-fold results for {best}:")
fold_df[["fold", "f1", "precision", "recall", "auroc"]].round(3)

---
## Step 5: Use the ML Tag

After `AssignMLTag`, the predictions are stored as a composite tag — use `FetchTag` and `ListTags` just like any other tag.

In [ ]:
# Fetch the ML-generated tag for specific ASNs
# Known positives — should be True
# AS7018: AT&T; AS7922: Comcast; AS3320: Deutsche Telekom
print("=== Known Residential ISPs (should be True) ===")
for asn in ["7018", "7922", "3320"]:
    val = tagger.FetchTag("Residential ISP", asns=asn)
    print(f"  AS{asn}: {val}")

# Known negatives — should be False
# AS13335: Cloudflare; AS15169: Google; AS32934: Facebook (Meta)
print("\n=== Known CDN/Cloud/Enterprise (should be False) ===")
for asn in ["13335", "15169", "32934"]:
    val = tagger.FetchTag("Residential ISP", asns=asn)
    print(f"  AS{asn}: {val}")

In [ ]:
# Fetch for ASNs NOT in training set (true generalization)
# Residential access ASes in a large ISP: AS3215: Orange; AS9506: Singtel
# Non-residential access ASes (e.g., transit AS) in a large ISP: AS5511: Orange; AS6453: Tata; AS7473: Singtel, AS39832: Opera Norway
test_asns = ["3215", "9506", "5511", "6453", "7473", "39832"]
print("=== Predictions on unseen ASNs ===")
for asn in test_asns:
    if normalize_asn_input(asn) in eyeball_canon:
        val = tagger.FetchTag("Residential ISP", asns=asn)
        print(f"  AS{asn}: {val}")
    else:
        print(f"  AS{asn}: Not an eyeball AS")

In [ ]:
print(f"Total eyeball ASNs: {len(eyeball_asns)}")
all_predictions = tagger.FetchTag("Residential ISP")
print(f"Total ASNs tagged: {len(all_predictions)}")

In [ ]:
import json
with open("./data/residential_isp/classified_residential_eyeball_asns.json", "w") as f:
    json.dump(all_predictions, f)

In [ ]:
# The ML tag appears alongside preset tags in ListTags
tags = tagger.ListTags(7018)
print("Composite tags for AT&T (AS7018):")
for tag, val in tags['Composite'].items():
    print(f"  {tag}: {val}")

---
## Advanced Usage

`AssignMLTag` exposes several parameters you can customize. Below is a quick reference, followed by examples.

### AssignMLTag Parameter Reference

| Parameter | Type | Default | User-settable? | Description |
|-----------|------|---------|----------------|-------------|
| `tag_name` | str | — | **Required** | Name of the composite tag to create |
| `positive_asns` | List[str] | — | **Required** | ASNs known to have the target property |
| `negative_asns` | List[str] | — | **Required** | ASNs known NOT to have the target property |
| `models` | List[str] | None (all) | ✅ | Which models to try: `["xgboost", "mlp", "graphconv", "appnp"]` |
| `n_folds` | int | 5 | ✅ | Number of stratified CV folds |
| `metric` | str | `"f1"` | ✅ | Selection metric: `"f1"`, `"auroc"`, `"auprc"`, `"precision"`, `"recall"` |
| `threshold` | float | 0.5 | ✅ | Probability cutoff for binary tag (0–1) |
| `verbose` | bool | True | ✅ | Print progress to stdout |
| `log_dir` | str | None | ✅ | Directory path to save `log.txt` and `ml_training_log.json` |
| `features` | List[str] | None (all) | ✅ | Subset of feature names; unrecognized ones are skipped |
| `share_specs` | List[tuple] | DEFAULT | ✅ | Share features; see **Share Features** section at end |
| `drop_numerators` | bool | True | ✅ | If True, drop numerator cols; if False, keep both numerators and share outputs |
| `asns` | List[str] | None (all) | ✅ | Restrict predictions to these ASNs only |
| `keep_training_labels` | bool | False | ✅ | If True, labeled ASNs keep literal label instead of model prediction |
| `xgboost_profile` | str | None | ✅ | Force XGBoost profile: `"stochastic"`, `"default"`, `"deep"`, `"shallow"`; None = auto-select |

### Use Only Specific Models

**Set:** `models` — list of model names to try.

When unset (`None`), all available models run (xgboost, mlp, graphconv, appnp). Graph models require DGL + topology data. To skip graph models or speed up, pass only feature-based ones.

In [ ]:
MODEL_DIR = "./data/residential_isp_v2"

# User-settable: models, n_folds, metric, threshold
results_fast = tagger.AssignMLTag(
    tag_name="Residential ISP v2",
    positive_asns=positive_asns,
    negative_asns=negative_asns,
    models=["xgboost"],   # Restrict to these models; skip graph models
    n_folds=5,                   # 3 folds = faster (default 5)
    metric="auroc",              # Select best model by AUROC (default: f1)
    threshold=0.5,
    model_dir=MODEL_DIR,         # Save for ml_feature_importance.ipynb
)

print(f"\nBest model: {results_fast['best_model']}")
print(f"Model saved to: {MODEL_DIR}")
results_fast["cv_results"][["model", "f1_mean", "auroc_mean"]].round(3)

In [ ]:
MODEL_DIR = "./data/residential_isp_ipinfo"

# User-settable: models, n_folds, metric, threshold
results_fast = tagger.AssignMLTag(
    tag_name="Residential ISP v3",
    positive_asns=positive_asns,
    negative_asns=negative_asns,
    models=["xgboost"],   # Restrict to these models; skip graph models
    n_folds=5,                   # 3 folds = faster (default 5)
    metric="auroc",              # Select best model by AUROC (default: f1)
    threshold=0.5,
    model_dir=MODEL_DIR,         # Save for ml_feature_importance.ipynb
)

print(f"\nBest model: {results_fast['best_model']}")
print(f"Model saved to: {MODEL_DIR}")
results_fast["cv_results"][["model", "f1_mean", "auroc_mean"]].round(3)

In [ ]:
# Exclude entire feature sources by prefix (text before first "_")
EXCLUDE_SOURCES = {"censys", "merit", "hg_offnet"}

def feature_source(name: str) -> str:
    i = str(name).find("_")
    return name if i < 0 else name[:i]

# Reuse typed feature lists from the inspect cell (num_feats, cat_feats)
# or re-detect if that cell was not run:
if "num_feats" not in globals() or "cat_feats" not in globals():
    from as_tagging.ml.feature_engineering import (
        identify_feature_types,
        identify_feature_types_from_metadata,
    )
    manifest = snapshot_schema = None
    if hasattr(tagger.snapshot_provider, "get_manifest"):
        try:
            manifest = tagger.snapshot_provider.get_manifest(DATE)
        except Exception:
            pass
    if hasattr(tagger.snapshot_provider, "get_schema"):
        try:
            snapshot_schema = tagger.snapshot_provider.get_schema(DATE)
        except Exception:
            pass
    num_feats, cat_feats, _ = identify_feature_types_from_metadata(
        manifest=manifest, schema=snapshot_schema
    )
    if not num_feats and not cat_feats:
        num_feats, cat_feats = identify_feature_types(tagger.atomic_tags)

all_typed = num_feats + cat_feats
features_no_censys_merit = [
    f for f in all_typed if feature_source(f) not in EXCLUDE_SOURCES
]
excluded = [f for f in all_typed if feature_source(f) in EXCLUDE_SOURCES]
print(f"Using {len(features_no_censys_merit)} features (excluded {len(excluded)} from {sorted(EXCLUDE_SOURCES)})")
print("Remaining sources:", sorted({feature_source(f) for f in features_no_censys_merit}))

MODEL_DIR = "./data/residential_isp_no_censys_merit"

results_fast = tagger.AssignMLTag(
    tag_name="Residential ISP v4",
    positive_asns=positive_asns,
    negative_asns=negative_asns,
    models=["xgboost"],
    n_folds=5,
    metric="auroc",
    threshold=0.5,
    features=features_no_censys_merit,
    share_specs=[],              # required: default share specs are all Censys fractions
    model_dir=MODEL_DIR,
)

print(f"\nBest model: {results_fast['best_model']}")
print(f"Model saved to: {MODEL_DIR}")
results_fast["cv_results"][["model", "f1_mean", "auroc_mean"]].round(3)

In [ ]:
# Exclude entire feature sources by prefix (text before first "_")
EXCLUDE_SOURCES = {"censys", "merit", "hg_offnet", "apnic-eyeball", "mlab-ndt"}

def feature_source(name: str) -> str:
    i = str(name).find("_")
    return name if i < 0 else name[:i]

# Reuse typed feature lists from the inspect cell (num_feats, cat_feats)
# or re-detect if that cell was not run:
if "num_feats" not in globals() or "cat_feats" not in globals():
    from as_tagging.ml.feature_engineering import (
        identify_feature_types,
        identify_feature_types_from_metadata,
    )
    manifest = snapshot_schema = None
    if hasattr(tagger.snapshot_provider, "get_manifest"):
        try:
            manifest = tagger.snapshot_provider.get_manifest(DATE)
        except Exception:
            pass
    if hasattr(tagger.snapshot_provider, "get_schema"):
        try:
            snapshot_schema = tagger.snapshot_provider.get_schema(DATE)
        except Exception:
            pass
    num_feats, cat_feats, _ = identify_feature_types_from_metadata(
        manifest=manifest, schema=snapshot_schema
    )
    if not num_feats and not cat_feats:
        num_feats, cat_feats = identify_feature_types(tagger.atomic_tags)

all_typed = num_feats + cat_feats
features_no_censys_merit = [
    f for f in all_typed if feature_source(f) not in EXCLUDE_SOURCES
]
excluded = [f for f in all_typed if feature_source(f) in EXCLUDE_SOURCES]
print(f"Using {len(features_no_censys_merit)} features (excluded {len(excluded)} from {sorted(EXCLUDE_SOURCES)})")
print("Remaining sources:", sorted({feature_source(f) for f in features_no_censys_merit}))

MODEL_DIR = "./data/residential_isp_no_easy_to_lose"

results_fast = tagger.AssignMLTag(
    tag_name="Residential ISP v5",
    positive_asns=positive_asns,
    negative_asns=negative_asns,
    models=["xgboost"],
    n_folds=5,
    metric="auroc",
    threshold=0.5,
    features=features_no_censys_merit,
    share_specs=[],              # required: default share specs are all Censys fractions
    model_dir=MODEL_DIR,
)

print(f"\nBest model: {results_fast['best_model']}")
print(f"Model saved to: {MODEL_DIR}")
results_fast["cv_results"][["model", "f1_mean", "auroc_mean"]].round(3)

In [ ]:
classified_residential = tagger.FetchTag("Residential ISP v2")
classified_eyeball_residential = set(classified_residential).intersection(set(eyeball_asns))
print(len(classified_residential), len(classified_eyeball_residential))

In [ ]:
print(13335 in classified_residential, 13335 in classified_eyeball_residential)
print(39832 in classified_residential, 39832 in classified_eyeball_residential)
# print(395954 in classified_residential, 395954 in classified_eyeball_residential)
# print(27411 in classified_residential, 27411 in classified_eyeball_residential)
print(396362 in classified_residential, 396362 in classified_eyeball_residential)
print(21769 in classified_residential, 21769 in classified_eyeball_residential)
print("***")
print(3215 in classified_residential, 3215 in classified_eyeball_residential)
print(9506 in classified_residential, 9506 in classified_eyeball_residential)
print("***")
print(5511 in classified_residential, 5511 in classified_eyeball_residential)
print(6453 in classified_residential, 6453 in classified_eyeball_residential)
print(7473 in classified_residential, 7473 in classified_eyeball_residential)

In [ ]:
import json
with open("./data/residential_isp/classified_residential_eyeball_asns.json", "w") as f:
    json.dump(list(classified_eyeball_residential), f)

In [ ]:
print(len(eyeball_asns))
print(len(set(set(classified_residential).difference(set(eyeball_asns)))))

### Tag Only Selected ASNs + Save/Load Model + Control Labeled-ASN Behavior

**Set:** `asns` — restrict which ASNs get predictions/tags.  
**Set:** `model_dir` — save the trained model to a local directory (e.g. `./data/residential_isp`).  
**Set:** `model_path` — load a saved model and assign tags without retraining.  
**Set:** `keep_training_labels` — when True, ASNs in positive_asns/negative_asns keep their literal label (True/False) instead of model prediction.

- `asns=None` (default): predict and tag all ASNs in the snapshot  
- `asns=[...]`: predict only those ASNs (e.g. eyeball-only scope)  
- `model_dir="./data/..."`: save model and config after training for later use  
- `model_path="./data/..."`: skip training and assign tags from a previously saved model  
- `keep_training_labels=True`: labeled ASNs never overwritten by model output

In [ ]:
target_asns = ["7018", "7922", "13335", "15169", "3320"]
MODEL_DIR = "./data/residential_isp"  # Local path to save/load model

# Example 1: asns + model_dir — save trained model to disk for later use
results_targeted_saved = tagger.AssignMLTag(
    tag_name="Residential ISP (targeted + saved)",
    positive_asns=positive_asns,
    negative_asns=negative_asns,
    models=["xgboost"],
    n_folds=3,
    asns=target_asns,
    model_dir=MODEL_DIR,  # Saves config.json + model.xgb (or model.pt) to ./data/residential_isp
)

In [ ]:
# Example 2: model_path — load saved model and assign tags without retraining
tagger.AssignMLTag(
    tag_name="Residential ISP (from saved model)",
    model_path=MODEL_DIR,
    asns=target_asns,
)

# # Example 3: asns + keep_training_labels — scope + literal labels for train ASNs
# results_targeted_keep_labels = tagger.AssignMLTag(
#     tag_name="Residential ISP (targeted + keep labels)",
#     positive_asns=positive_asns,
#     negative_asns=negative_asns,
#     models=["xgboost"],
#     n_folds=3,
#     asns=target_asns,
#     keep_training_labels=True,  # pos/neg ASNs keep True/False, not model output
# )

print("Targeted predictions (from saved model):")
for asn in target_asns:
    val = tagger.FetchTag("Residential ISP (from saved model)", asns=asn)
    print(f"  AS{asn}: {val}")

### Custom Feature Selection

**Set:** `features` — list of feature names to use. Default `None` = all auto-detected numerical + categorical.

- Use only features present in your snapshot (check schema or atomic_tags keys)  
- Unrecognized or unsupported types (list/dict) are skipped with a warning  
- Features are from snapshot atomic tags (e.g. `pfx2as_/24_cnt`, `delegation_rir`)

In [ ]:
# User-settable: features
results_custom = tagger.AssignMLTag(
    tag_name="Residential ISP (custom features)",
    positive_asns=positive_asns,
    negative_asns=negative_asns,
    models=["xgboost"],
    n_folds=3,
    features=[  # Subset of features; nonexistent ones skipped with warning
        # Numerical features
        "apnic-eyeball_cc_cnt",
        "caida-asrel_customer_cnt",
        "caida-asrel_peer_cnt",
        "caida-asrel_provider_cnt",
        "pfx2as_/24_cnt",
        # Categorical features
        "delegation_rir",
        # These will be SKIPPED with a warning:
        "nonexistent_feature",           # does not exist in snapshot
        "caida-asrel_customer_list",      # list type — excluded
    ],
)

print(f"\nBest model: {results_custom['best_model']}")
results_custom["cv_results"][["model", "f1_mean", "auroc_mean"]].round(3)

### Save Training Logs to Disk

**Set:** `log_dir` — directory path to save logs. When set, creates:
- **`log.txt`** — streaming text log (same as stdout)
- **`ml_training_log.json`** — structured results (config, features, CV metrics, labeled ASNs)

In [ ]:
# User-settable: log_dir
results_logged = tagger.AssignMLTag(
    tag_name="Residential ISP (logged)",
    positive_asns=positive_asns,
    negative_asns=negative_asns,
    models=["xgboost"],
    n_folds=3,
    log_dir="./ml_logs/residential",  # Saves log.txt and ml_training_log.json
)

### XGBoost Profile

**Set:** `xgboost_profile` — force XGBoost hyperparameters instead of inner-CV selection.

- `None` (default): auto-select via inner CV among 4 profiles  
- `"stochastic"`: matches old pipeline; reduces fold variance  
- `"default"`, `"deep"`, `"shallow"`: other predefined profiles

(For `share_specs`, see the **Share Features** section at the end of this notebook.)

In [ ]:
# Inspect the saved structured log
import json

with open("./ml_logs/residential/ml_training_log.json") as f:
    log = json.load(f)

print(f"Timestamp: {log['timestamp']}")
print(f"Best model: {log['best_model']}")
print(f"Features: {log['features']['n_numerical']} numerical, "
      f"{log['features']['n_categorical']} categorical")
print(f"\nCV Summary:")
for row in log['cv_summary']:
    print(f"  {row['model']}: F1={row['f1_mean']:.3f} ± {row['f1_std']:.3f}")

# Also peek at the text log
print(f"\n--- log.txt (first 5 lines) ---")
with open("./ml_logs/residential/log.txt") as f:
    for i, line in enumerate(f):
        if i >= 5: break
        print(line, end='')

### Low-Level API: MLTagger Directly

For raw probabilities or finer control, use `MLTagger` instead of `AssignMLTag`.  

**MLTagger constructor** accepts: `snapshot_dict`, `manifest`, `snapshot_schema`, `models`, `verbose`, `log_dir`, `features`, `share_specs`, `xgboost_profile` (same semantics as AssignMLTag).  

**Methods:** `train_and_select()`, `predict(asns)`, `tag(threshold, asns, keep_training_labels, ...)`. Use `tagger.AssignTag(name, dict)` to register predictions as a composite tag.

In [ ]:
from as_tagging.ml import MLTagger

# MLTagger: same user-settable params as AssignMLTag (models, features, share_specs, etc.)
ml = MLTagger(
    snapshot_dict=tagger.atomic_tags,
    models=["xgboost"],
    verbose=True,
    # Optional: features=..., share_specs=..., xgboost_profile="stochastic", log_dir=...
)

# Train and cross-validate
results = ml.train_and_select(
    positive_asns=positive_asns,
    negative_asns=negative_asns,
    n_folds=5,
    metric="f1",
)

In [ ]:
# ml.predict(asns) — returns {asn: probability}; asns=None = all
probs = ml.predict(asns=["7018", "7922", "13335", "15169", "3320"])
print("Raw probabilities:")
for asn, p in sorted(probs.items(), key=lambda x: -x[1]):
    print(f"  AS{asn}: {p:.4f}")

In [ ]:
# ml.tag(threshold, asns, keep_training_labels, positive_asns, negative_asns)
tags_strict = ml.tag(threshold=0.7)  # User-settable: stricter cutoff
tags_loose = ml.tag(threshold=0.3)   # More permissive

n_strict = sum(1 for v in tags_strict.values() if v)
n_loose = sum(1 for v in tags_loose.values() if v)
print(f"Threshold 0.7: {n_strict} ASNs tagged positive")
print(f"Threshold 0.3: {n_loose} ASNs tagged positive")

# Same params as AssignMLTag: asns, keep_training_labels, positive_asns, negative_asns
tags_subset_keep = ml.tag(
    threshold=0.5,
    asns=["7018", "7922", "13335", "15169", "3320"],
    keep_training_labels=True,
    positive_asns=positive_asns,
    negative_asns=negative_asns,
)
print("\nSubset tags (keep training labels):", tags_subset_keep)

In [ ]:
# Assign the predictions as a tag manually
tags_default = ml.tag(threshold=0.5)
tagger.AssignTag("Residential ISP (manual)", tags_default)

# Verify
print(tagger.FetchTag("Residential ISP (manual)", asns="7018"))

---
## Data Format Reference

The ML module works with the `atomic_tags` dictionary from `ASTagging`. The format is:

```python
tagger.atomic_tags = {
    "7018": {
        "pfx2as_/24_cnt": 1532.0,
        "apnic-eyeball_users": 22080.0,
        "caida-asrel_customer_cnt": 3,
        "caida-asrel_customer_list": '["123", "456"]',
        "delegation_rir": "arin",
        ...
    },
    "13335": { ... },
    ...
}
```

### Feature Types (auto-detected)

| Type | Example Features | Handling |
|---|---|---|
| **Numerical** | `pfx2as_/24_cnt`, `apnic-eyeball_users` | NaN -> 0, passed directly |
| **Categorical** | `delegation_rir`, `peeringdb_info_type` | Label-encoded + embeddings |
| **List/JSON** | `caida-asrel_customer_list` | Skipped (not used as features) |

### Graph Construction

Graph-based models (GraphConv, APPNP) require CAIDA AS-relationship features:
- `caida-asrel_customer_list` — for customer edges
- `caida-asrel_peer_list` — for peer edges  
- `caida-asrel_provider_list` — for provider edges

If these features are missing from the snapshot, graph models are **automatically skipped**.

### Share Features (`share_specs`)

**What they are:** Share features are Laplace-smoothed fractions `(num + alpha) / (denom + K*alpha)` computed from raw counts. Each spec is a tuple `(numerator_col, denominator_col, output_col, alpha, K)`.

**Default spec** (`share_specs=None`): The built-in `DEFAULT_SHARE_SPECS` computes 9 fraction features from Censys top-3 port, OS, and service counts. Numerators (e.g. `censys_port_1_cnt`) are dropped from the final feature set to avoid redundancy.

| numerator | denominator | output |
|-----------|-------------|--------|
| `censys_port_1_cnt` | `censys_v4addr_cnt` | `censys_port_1_frac` |
| `censys_port_2_cnt` | `censys_v4addr_cnt` | `censys_port_2_frac` |
| `censys_port_3_cnt` | `censys_v4addr_cnt` | `censys_port_3_frac` |
| `censys_os_1_cnt` | `censys_v4addr_cnt` | `censys_os_1_frac` |
| `censys_os_2_cnt` | `censys_v4addr_cnt` | `censys_os_2_frac` |
| `censys_os_3_cnt` | `censys_v4addr_cnt` | `censys_os_3_frac` |
| `censys_service_1_cnt` | `censys_v4addr_cnt` | `censys_service_1_frac` |
| `censys_service_2_cnt` | `censys_v4addr_cnt` | `censys_service_2_frac` |
| `censys_service_3_cnt` | `censys_v4addr_cnt` | `censys_service_3_frac` |

*(All use `alpha=1.0`, `K=2`.)*

**Setting `share_specs`:**
- `None` (default): use the table above
- `[]`: disable share features; keep raw count features (e.g. `censys_port_1_cnt`)
- Custom list: define your own `(num, denom, out, alpha, K)` tuples

**Setting `drop_numerators`:**
- `True` (default): drop numerator cols (e.g. `censys_port_1_cnt`) after computing shares; final features = base + share outputs only
- `False`: keep both numerators and share outputs in the feature set


In [ ]:
# Example: disable share features (use raw counts only)
# tagger.AssignMLTag(..., share_specs=[])

# Example: keep numerators — both censys_port_1_cnt and censys_port_1_frac in features
# tagger.AssignMLTag(..., drop_numerators=False)

# Example: custom share specs
# custom_specs = [("my_cnt", "my_total_cnt", "my_frac", 1.0, 2)]
# tagger.AssignMLTag(..., share_specs=custom_specs)